In [1]:
from pathlib import Path
from eeg_music.data import EEGMusicDataset, MappedDataset, MelParams, TrialData, WavRAW, prepare_trial, wavraw_to_melspectrogram


# ds = EEGMusicDataset.load_ondisk(
#     Path("./datasets/bcmi_preprocessed/bcmi_pre_60ch/")
#   )
ds = EEGMusicDataset.load_ondisk(
    Path("./datasets/musing_preprocessed/musing_wav_60ch/")
  )

def applying_mel(trial, apply_mel: MelParams, wav_resample=None):
  music = trial.music_data.get_music()
  match music:
    case WavRAW(raw_data=raw, sample_rate=sr) as wav:
      if wav_resample is not None and wav_resample != sr:
        wav = wav.resampled(new_sr=wav_resample)
        # raw, sr = wav.raw_data, wav.sample_rate
      music_cropped = wavraw_to_melspectrogram(wav, **apply_mel.as_kwargs())
      return TrialData(
        dataset=trial.dataset,
        subject=trial.subject,
        session=trial.session,
        run=trial.run,
        trial_id=trial.trial_id,
        music_filename=trial.music_filename,
        eeg_data=trial.eeg_data,
        music_data=music_cropped,
      )
    case _:
      raise ValueError("Unexpected music data format")

mds = MappedDataset(ds, lambda t: applying_mel(
    t,
    wav_resample=32768,
    apply_mel=MelParams(n_mels=64, fmax=8192.0),
  ))


/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [4]:
mds[0].eeg_data.get_array().data.shape, mds[0].music_data.mel.shape

((60, 1351), (64, 29007))

In [5]:
# mds.save(Path("./datasets/bcmi_preprocessed/bcmi_mel64_60ch/"))
mds.save(Path("./datasets/musing_preprocessed/musing_mel64_60ch/"))

In [27]:
from fractions import Fraction
from eeg_music.data import ArrayStratifiedSamplingDataset


strat = ArrayStratifiedSamplingDataset(mds, 10, trial_length_secs=Fraction(2, 1))
strat[0].eeg_data.get_array().data.shape, strat[0].music_data.mel.shape

((60, 20), (64, 128))

In [30]:
strat[0].music_data.mel[:, 32:64+32].shape

(64, 64)

In [6]:
mds1 = EEGMusicDataset.load_ondisk(Path("./datasets/musing_preprocessed/musing_mel64_60ch/"))
mds1[0].music_data.get_music().mel

array([[-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.],
       ...,
       [-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.]], dtype=float32)

In [8]:
mel_arrays = [music_data.music_data.get_music().mel for music_data in mds1]
min_val = min(arr.min() for arr in mel_arrays)
max_val = max(arr.max() for arr in mel_arrays)
print(f"Min: {min_val}, Max: {max_val}")

Min: -80.0, Max: 3.814697265625e-06
